# **<font color="red">Plugins</font>**
A Plugin in Agent Developement Kit (ADK) is a custom code module that can be executed at various stages of an agent workflow lifecycle using callback hooks. You use Plugins for functionality that is applicable across your agent workflow. Some typical applications of Plugins are as follows:
- **Logging and Tracing:** Create detailed logs of agent, tool, and generative AI model activity for debugging and performance analysis.
- **Policy Enforcement:** Implement security guardrails, such as a function that checks if users are authorized to use a specific tool and prevent its execution if they do not have permission.
- **Monitoring and Metrics:** Collect and export metrics on token usage, execution times, and invocation counts to monitoring systems sucha s Prometheus or ***Google Cloud Observability*** (formerly Stackdriver).
- **Response Caching:** Check if a request has been made before, so you can return a cached response, skipping expensive or time consuming AI model or tool calls.
- **Request or Response Modification:** Dynamically add information to AI model prompts or standardize tool output responses.

## **<font color="blue">How do Plugins Work?</font>**
- An ADK Plugin extends the `BasePlugin` class and contains one or more `callback` methods, indicating where in the agent lifecycle the Plugin should be executed.
- You integrate Plugins into an agent by registering them in your agent's `Runner` class.
- Plugin functionality builds on <u>Callbacks</u>, which is a key design element of the ADK's extensible architecture.
- While a typical Agent Callback is configured on a _single agent_, a _single tool_ for a _specific task_, a Pluging is registered _once_ on the `Runner` and its callbacks apply _globally_ to every agent, tool, and LLM call managed by the runner.
- Plugins let you package related callback functions together to be used across a workflow. This makes Plugins an ideal solution for implementing features that cut across your entire agent application.

## **<font color="blue">Prebuilt Plugins</font>**
ADK includes several plugins that you can add to your agent workflow immediately:

### **<font color="green">Reflect and Retry Tools:</font>**
- The Reflect and Retry plugin can help your agent recover from error responses from ADK Tools and automatically retry the tool request.
- This plugin intercepts tool failures, provides structured guidance to the AI model for reflection and correction, and retries the operation up to a configurable limit.
- This plugin can help you build more resilience into your agent workflows, including the following capabilities:
  - **Concurrency Safe:** Uses locking to safely handle parallel tool executions.
  - **Configurable Scope:** Tracks failures pre-invocation (default) or globally.
  - **Granular Tracking:** Failure counts are tracked per-tool
  - **Custome Error Extraction:** Supports detecting errors in normal tool responses.
#### **Add Reflect and Retry Plugin**
- Add this plugin to your ADK workflow by adding it to the plugins setting of your ADK project's.
```python
from google.adk.apps.app import App
from google.adk.plugins import ReflectAndRetryToolPlugin

app = App(
    name="my_app",
    root_agent=root_agent,
    plugins=[
        ReflectAndRetryToolPlugin(max_retries=3),
    ],
)
```

In [9]:
import os
import asyncio
from datetime import datetime
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.plugins import ReflectAndRetryToolPlugin
from google.adk.tools import FunctionTool
import google.genai.types as types
from config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = "gemini-2.5-flash"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Define Tool: get_current_datetime
# -----------------------------
def get_current_datetime() -> dict:
    now = datetime.now()
    return {
        "date": now.strftime("%d/%m/%Y"),
        "time": now.strftime("%H:%M:%S"),
        "day": now.strftime("%A")
    }

get_current_datetime_tool = FunctionTool(
    func=get_current_datetime
)

# -----------------------------
# Create LlmAgent with Tool
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="ChatAgent",
    instruction="""
    You are a helpful assistant. Respond politely and concisely to user questions.
    Use available tools when needed.
    """,
    tools=[get_current_datetime_tool]
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# -----------------------------
# Create a new session
# -----------------------------
async def create_session():
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

# -----------------------------
# Setup Runner with ReflectAndRetry Plugin
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[ReflectAndRetryToolPlugin(max_retries=3)]
)

# -----------------------------
# Chat function
# -----------------------------
async def chat(user_input):
    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )
    
    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
    
    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            print("ChatAgent:", event.content.parts[0].text)

# -----------------------------
# Main async flow with 5 conversations
# -----------------------------
async def main():
    await create_session()
    
    # Conversation 1: Simple greeting
    await chat("Hello! How are you today?")
    
    # Conversation 2: Ask for a weekend activity
    await chat("Can you suggest a fun weekend activity?")
    
    # Conversation 3: Call the get_current_datetime tool
    await chat("Please run get_current_datetime for me.")
    
    # Conversation 4: Ask a general knowledge question
    await chat("Can you give me a short summary of Artificial Intelligence?")
    
    # Conversation 5: Say goodbye
    await chat("Thanks for your help! Goodbye!")

# Run in Jupyter Lab
await main()


C:\Users\DELL\AppData\Local\Temp\ipykernel_6752\2333852231.py:75: UserWarning: [EXPERIMENTAL] ReflectAndRetryToolPlugin: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  plugins=[ReflectAndRetryToolPlugin(max_retries=3)]


ChatAgent: Hello! I'm doing well, thank you for asking. How can I help you today?
ChatAgent: How about exploring a local park or nature trail? It's a great way to enjoy the outdoors and get some exercise.
ChatAgent: The current date is Thursday, February 27, 2026, and the time is 12:51:54.
ChatAgent: Artificial Intelligence (AI) is a broad field of computer science that focuses on creating machines that can perform tasks that typically require human intelligence. This includes things like learning, problem-solving, decision-making, understanding language, and recognizing patterns. The goal of AI is to enable computers to think and act like humans, or even surpass human capabilities in certain areas.
ChatAgent: You're welcome! Goodbye, and have a great day!


#### **Configuration Settings**
- The Reflect and Retry Plugin has the following configuraion options:
  - **`max_retries` (optional):** Total number of additional attempts the system makes to receive a non-error response. Default value is 3.
  - **`throw_exception_if_retry_exceeded`(optional):** If set to `False`, the system does not raise an error if the final retry attempts fails. Default value is `True`.
  - **`tracking_scope` (optional):**
    - **`TrackingScope.INVOCATION`:** Track tool failures across a single invocation and user. This value is the default.
    - **`TrackingScope.GLOBAL`:** Track tool failures across all invocations and all users.
- **Advanced Configuration:** You can further modify the behavior of this plugin by extending the `ReflectAndRetryToolPlugin` class.
```python
class CustomRetryPlugin(ReflectAndRetryToolPlugin):
  async def extract_error_from_result(self, *, tool, tool_args,tool_context,
  result):
    # Detect error based on response content
    if result.get('status') == 'error':
        return result
    return None  # No error detected

# add this modified plugin to your App object:
error_handling_plugin = CustomRetryPlugin(max_retries=5)
```

In [17]:
import os
import asyncio
import random
from datetime import datetime
from typing import Any

from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.plugins import ReflectAndRetryToolPlugin
from google.adk.plugins.reflect_retry_tool_plugin import TrackingScope
from google.adk.tools import FunctionTool
import google.genai.types as types
from config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = "gemini-2.5-flash"

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Custom ReflectAndRetry Plugin with logging
# -----------------------------
class CustomRetryPlugin(ReflectAndRetryToolPlugin):
    async def extract_error_from_result(
        self, *, tool, tool_args, tool_context, result
    ):
        if isinstance(result, dict) and result.get("status") == "error":
            attempt = getattr(tool_context, "retry_attempt", 1)
            print(f"[Retry Attempt {attempt}] Tool '{tool.name}' failed: {result}")
            return result
        return None

# Instantiate plugin
error_handling_plugin = CustomRetryPlugin(
    max_retries=5,
    throw_exception_if_retry_exceeded=False,
    tracking_scope=TrackingScope.GLOBAL
)

# -----------------------------
# Define Tool 1: get_current_datetime
# -----------------------------
def get_current_datetime() -> dict:
    if random.random() < 0.3:  # 30% chance to "fail"
        return {"status": "error", "message": "Simulated failure"}
    
    now = datetime.now()
    return {
        "status": "success",
        "date": now.strftime("%d/%m/%Y"),
        "time": now.strftime("%H:%M:%S"),
        "day": now.strftime("%A")
    }

get_current_datetime_tool = FunctionTool(func=get_current_datetime)

# -----------------------------
# Define Tool 2: guess_number_tool
# -----------------------------
def guess_number_tool(query: int) -> dict[str, Any]:
    target_number = 3
    if query == target_number:
        return {"status": "success", "result": "Number is valid."}
    if abs(query - target_number) <= 2:
        return {"status": "error", "error_message": "Number is almost valid."}
    if query > target_number:
        raise ValueError("Number is too large.")
    if query < target_number:
        raise ValueError("Number is too small.")
    raise ValueError("Number is invalid.")

guess_number_tool = FunctionTool(func=guess_number_tool)

# -----------------------------
# Create LlmAgent with Tools
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="ChatAgent",
    instruction="""
    You are a helpful assistant. Respond politely and concisely to user questions.
    Use available tools when needed.
    """,
    tools=[get_current_datetime_tool, guess_number_tool]
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# -----------------------------
# Create a new session
# -----------------------------
async def create_session():
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

# -----------------------------
# Setup Runner with Custom Plugin
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[error_handling_plugin]
)

# -----------------------------
# Chat function with Human/AI labels
# -----------------------------
async def chat(user_input: str):
    print("Human:", user_input)
    
    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )
    
    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
    
    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            print("AI:", event.content.parts[0].text, "\n")

# -----------------------------
# Main async flow with 5 conversations
# -----------------------------
async def main():
    await create_session()
    
    await chat("Hello! How are you today?")
    await chat("Can you suggest a fun weekend activity?")
    await chat("Please run get_current_datetime for me.")
    await chat("Give me a short summary of Artificial Intelligence.")
    await chat("I want to guess the number 2. Can you check if it's correct?")

# Run in Jupyter Lab
await main()

C:\Users\DELL\AppData\Local\Temp\ipykernel_6752\2598775177.py:41: UserWarning: [EXPERIMENTAL] ReflectAndRetryToolPlugin: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  error_handling_plugin = CustomRetryPlugin(


Human: Hello! How are you today?
AI: I'm doing well, thank you for asking! How can I help you today? 

Human: Can you suggest a fun weekend activity?
AI: I'd love to help with that! To give you the best suggestion, could you tell me a little more about what you enjoy? For example, do you prefer indoor or outdoor activities, relaxing or adventurous, or something for a specific group size? 

Human: Please run get_current_datetime for me.
[Retry Attempt 1] Tool 'get_current_datetime' failed: {'status': 'error', 'message': 'Simulated failure'}
AI: The current date is Friday, 27/02/2026, and the time is 13:54:27.
 

Human: Give me a short summary of Artificial Intelligence.
AI: Artificial Intelligence (AI) is a broad field of computer science that enables machines to perform tasks that typically require human intelligence. This includes learning, problem-solving, perception, and decision-making. AI aims to create systems that can reason, understand, and interact with the world in intelligen